In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from model import AgeClassCNN
from src.dataset import get_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando no dispositivo {device}")

DATA_DIR = 'dataset_idades/'
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

In [ ]:
train_loader, val_loader, classes = get_dataloaders(DATA_DIR, BATCH_SIZE)
print(f"Classes encontradas: {classes}")

model = AgeClassCNN(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss
optimazer = optim.Adam(model.parameters(), LEARNING_RATE)

In [ ]:
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for inputs, labels_target in train_loader:
        inputs, labels_target = inputs.to(device), labels_target.to(device)

        optimazer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels_target)

        loss.backward()
        optimazer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad(): 
        for inputs, labels_target in val_loader:
            input, labels_target = inputs.to(device), labels_target.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels_target)
            val_loss += loss.item()

            _, labels_predicted = torch.max(outputs.data, 1)
            total += labels_target.size(0)
            correct += (labels_predicted == labels_target).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    accuracy = 100 * correct / total

    print(f"Época [{epoch + 1}/{EPOCHS}] | "
          f"Loss Treino: {avg_train_loss:.4f} | "
          f"Loss Validação: {avg_val_loss:.4f} | "
          f"Acurácia validação: {accuracy:.2f}%")

print("Treinamento finalizado!")
